# SDC M&A — Deal Counts, PE vs Strategic, Public vs Private

Annual US M&A transaction analysis from SDC Mergers & Acquisitions database.

**Data source:** SDC M&A (`tfn.sdc_ma` — schema must be discovered first)

**Reference:** `references/sdc-ma.md`

## Contents
1. Schema discovery
2. Annual M&A deal counts (completed, US targets)
3. PE/LBO vs strategic buyer split
4. Public vs private target breakdown
5. Deal value distribution
6. Summary: M&A practice area sizing

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

conn = psycopg2.connect(
    host='wrds-pgdata.wharton.upenn.edu',
    port=9737,
    database='wrds',
    user='eddyhu',
    sslmode='require'
)
print('Connected to WRDS')

## 1. Schema Discovery

In [ ]:
cur = conn.cursor()

# Find SDC schemas
cur.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name ILIKE '%sdc%' OR schema_name ILIKE '%tdc%'
    ORDER BY schema_name
""")
schemas = cur.fetchall()
print('SDC schemas:', schemas)

SDC_SCHEMA = schemas[0][0] if schemas else 'tfn'
print(f'Using schema: {SDC_SCHEMA}')

In [ ]:
# Find M&A table
cur.execute("""
    SELECT table_name,
           pg_size_pretty(pg_total_relation_size(
               quote_ident(table_schema)||'.'||quote_ident(table_name))) AS size
    FROM information_schema.tables
    WHERE table_schema = %s
    ORDER BY table_name
""", (SDC_SCHEMA,))
tables = cur.fetchall()
print('Tables:', tables)

MA_TABLE = f'{SDC_SCHEMA}.sdc_ma'  # update based on discovery
print(f'M&A table: {MA_TABLE}')

In [ ]:
# Inspect M&A columns
ma_table_name = MA_TABLE.split('.')[-1]
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s
    ORDER BY ordinal_position
""", (SDC_SCHEMA, ma_table_name))
cols = cur.fetchall()
print(f'Columns in {MA_TABLE}:')
for c in cols:
    print(f'  {c[0]:40s} {c[1]}')

In [ ]:
# Row count and date range
cur.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        COUNT(DISTINCT deal_no)         AS unique_deals,
        MIN(date_announced)             AS earliest,
        MAX(date_announced)             AS latest
    FROM {MA_TABLE}
    WHERE target_nation = 'United States'
""")
row = cur.fetchone()
print(f'US target deals: {row[1]:,.0f}')
print(f'Date range:      {row[2]} → {row[3]}')

## 2. Annual M&A Deal Counts

In [ ]:
# Pull completed US M&A deals
query_ma = f"""
SELECT
    deal_no,
    date_announced,
    date_effective,
    status,
    acquiror_name,
    target_name,
    acquiror_nation,
    target_nation,
    acquiror_public     AS apub,
    target_public       AS tpub,
    deal_value          AS tv,
    form_of_transaction,
    consideration_paid,
    pct_acquired,
    lbo,
    acquiror_type,
    attitude,
    premium_4wk,
    target_cusip,
    acquiror_sic,
    target_sic
FROM {MA_TABLE}
WHERE target_nation = 'United States'
  AND status IN ('C', 'W')
  AND date_announced BETWEEN '1985-01-01' AND '2024-12-31'
"""
df = pd.read_sql(query_ma, conn)
df['announce_year'] = pd.to_datetime(df['date_announced']).dt.year
df['tv'] = pd.to_numeric(df['tv'], errors='coerce')
df['pct_acquired'] = pd.to_numeric(df['pct_acquired'], errors='coerce')

print(f'Total US deals (completed+withdrawn): {len(df):,}')
print(f'Status breakdown:')
print(df['status'].value_counts())
df.head(3)

In [ ]:
# Clean sample: completed, majority acquisitions
FULL_CONTROL_FORMS = {
    'Merger', 'Acq. of Majority Interest',
    'Acq. of Remaining Interest', 'Asset Acquisition'
}

df_comp = df[
    (df['status'] == 'C') &
    (df['form_of_transaction'].isin(FULL_CONTROL_FORMS) | df['form_of_transaction'].isna()) &
    (df['pct_acquired'].isna() | (df['pct_acquired'] >= 50))
].copy()

print(f'Clean completed deals: {len(df_comp):,}')

# Annual counts and volume
annual = df_comp.groupby('announce_year').agg(
    n_deals=('deal_no', 'count'),
    n_with_value=('tv', lambda x: x.notna().sum()),
    total_value_bn=('tv', lambda x: x.sum() / 1e3),
    avg_value_mm=('tv', 'mean')
)
annual = annual[annual.index.between(1985, 2024)]
print(annual.tail(10))

In [ ]:
# Plot: annual deal count
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].bar(annual.index, annual['n_deals'], color='steelblue', alpha=0.85)
axes[0].set_ylabel('Number of Deals')
axes[0].set_title('US M&A: Annual Completed Deals (SDC)')

axes[1].bar(annual.index, annual['total_value_bn'], color='coral', alpha=0.85)
axes[1].set_ylabel('Total Value ($Bn)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig('/tmp/ma_annual.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. PE/LBO vs Strategic Buyer

In [ ]:
# Check PE-related column values
print('LBO flag distribution:')
print(df_comp['lbo'].value_counts(dropna=False).head(10))
print()
print('Acquiror type top values:')
print(df_comp['acquiror_type'].value_counts(dropna=False).head(15))

In [ ]:
# PE identification: LBO flag OR financial sponsor acquiror types
# Adjust PE_ACQUIROR_TYPES based on actual values seen above
PE_ACQUIROR_TYPES = {'PE', 'VC', 'I', 'SP'}  # update after inspecting output above

PE_NAME_PATTERNS = (
    r'private equity|buyout|capital partners|equity partners|'
    r'apollo|blackstone|carlyle|kkr|tpg|warburg pincus|bain capital|'
    r'advent|vista equity|thoma bravo|francisco partners|silver lake|'
    r'cerberus|ares management|oaktree|bc partners|apax|'
    r'general atlantic|insight partners|summit partners'
)

df_comp['pe_lbo_flag'] = df_comp['lbo'] == 'Yes'
df_comp['pe_type_flag'] = df_comp['acquiror_type'].isin(PE_ACQUIROR_TYPES)
df_comp['pe_name_flag'] = df_comp['acquiror_name'].str.lower().str.contains(
    PE_NAME_PATTERNS, na=False, regex=True)

df_comp['is_pe'] = (
    df_comp['pe_lbo_flag'] |
    df_comp['pe_type_flag'] |
    df_comp['pe_name_flag']
)

print(f'PE deals:        {df_comp["is_pe"].sum():,} ({df_comp["is_pe"].mean():.1%})')
print(f'Strategic deals: {(~df_comp["is_pe"]).sum():,} ({(~df_comp["is_pe"]).mean():.1%})')

In [ ]:
# Annual PE vs strategic split
pe_annual = (df_comp.groupby(['announce_year', 'is_pe'])
             .agg(n_deals=('deal_no', 'count'),
                  total_value_bn=('tv', lambda x: x.sum() / 1e3))
             .reset_index())

pe_pivot = pe_annual.pivot_table(
    index='announce_year', columns='is_pe',
    values=['n_deals', 'total_value_bn'], aggfunc='sum'
).fillna(0)
pe_pivot.columns = ['strategic_deals', 'pe_deals', 'strategic_value', 'pe_value']
pe_pivot['pe_share_pct'] = (
    pe_pivot['pe_deals'] /
    (pe_pivot['pe_deals'] + pe_pivot['strategic_deals']) * 100
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

pp = pe_pivot[pe_pivot.index.between(1990, 2024)]

axes[0].bar(pp.index, pp['strategic_deals'], label='Strategic', color='steelblue', alpha=0.85)
axes[0].bar(pp.index, pp['pe_deals'], bottom=pp['strategic_deals'],
            label='PE/LBO', color='coral', alpha=0.85)
axes[0].set_ylabel('Deal Count')
axes[0].set_title('US M&A: PE/LBO vs Strategic Buyers')
axes[0].legend()

axes[1].plot(pp.index, pp['pe_share_pct'], color='darkred', linewidth=1.5,
             marker='o', markersize=3)
axes[1].axhline(pp['pe_share_pct'].mean(), color='gray', linestyle='--',
                alpha=0.7, label=f'Mean: {pp["pe_share_pct"].mean():.0f}%')
axes[1].set_ylabel('PE Share (%)')
axes[1].set_xlabel('Year')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/ma_pe_strategic.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Public vs Private Target

In [ ]:
# Public/private distribution
print('Target public flag distribution:')
print(df_comp['tpub'].value_counts(dropna=False).head(10))

In [ ]:
# Annual: public target vs private target
pub_private = df_comp.groupby(['announce_year', 'tpub']).agg(
    n_deals=('deal_no', 'count'),
    total_value_bn=('tv', lambda x: x.sum() / 1e3)
).reset_index()

pub_pvt_pivot = pub_private.pivot_table(
    index='announce_year', columns='tpub',
    values=['n_deals', 'total_value_bn'], aggfunc='sum'
).fillna(0)

# Map P=public, V=private, S=subsidiary
print('Target types:', pub_pvt_pivot.columns.get_level_values(1).unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pp = pub_pvt_pivot[pub_pvt_pivot.index.between(1990, 2024)]
n = pp['n_deals']
v = pp['total_value_bn']

# Count
ax = axes[0]
colors = {'P': '#d62728', 'V': '#1f77b4', 'S': '#2ca02c'}
bottom = np.zeros(len(n))
for ttype in ['P', 'V', 'S']:
    if ttype in n.columns:
        ax.bar(n.index, n[ttype], bottom=bottom, label=ttype, color=colors.get(ttype,'gray'), alpha=0.85)
        bottom += n[ttype].values
ax.set_title('Deal Count by Target Type')
ax.set_ylabel('Number of Deals')
ax.legend(title='Target')

# Value
ax = axes[1]
bottom = np.zeros(len(v))
for ttype in ['P', 'V', 'S']:
    if ttype in v.columns:
        ax.bar(v.index, v[ttype], bottom=bottom, label=ttype, color=colors.get(ttype,'gray'), alpha=0.85)
        bottom += v[ttype].values
ax.set_title('Deal Value by Target Type ($Bn)')
ax.set_ylabel('Total Value ($Bn)')

plt.tight_layout()
plt.savefig('/tmp/ma_public_private.png', dpi=150, bbox_inches='tight')
plt.show()
print()
print('NOTE: Public targets = ~5-8% of deal count but ~40-60% of dollar volume.')
print('Public-target deals require proxy/tender offer filings — intensive legal work.')

## 5. Summary Statistics

In [ ]:
# Recent 5-year averages
recent = df_comp[df_comp['announce_year'].between(2018, 2022)]

print('=== M&A Practice Area Summary (2018–2022 average) ===')
print(f'Total completed US deals/year:    {len(recent)/5:,.0f}')
print(f'  PE/LBO deals/year:              {recent["is_pe"].sum()/5:,.0f} ({recent["is_pe"].mean():.1%} of total)')
print(f'  Strategic deals/year:           {(~recent["is_pe"]).sum()/5:,.0f}')
print(f'  Public target deals/year:       {(recent["tpub"]=="P").sum()/5:,.0f}')
print(f'  Deals w/ disclosed value:       {recent["tv"].notna().sum()/5:,.0f}')
print(f'  Avg disclosed deal value:       ${recent["tv"].mean():,.0f}M')
print()
print('Deal size distribution (disclosed values):')
bins = [0, 25, 100, 500, 1000, 5000, float('inf')]
labels = ['<$25M', '$25–100M', '$100–500M', '$500M–1Bn', '$1–5Bn', '>$5Bn']
recent_val = recent.dropna(subset=['tv'])
recent_val = recent_val[recent_val['tv'] > 0]
size_dist = pd.cut(recent_val['tv'], bins=bins, labels=labels).value_counts().sort_index()
for label, count in size_dist.items():
    print(f'  {label:15s}: {count/5:6.0f} deals/year')